In [ ]:
# Libraries
import numpy as np
import pandas as pd
import pickle
import re
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
#import data

In [ ]:
# Read the data to a pandas data frame
df = pd.read_csv('car-data.csv', sep=',', encoding='utf-8')
# Get number of rows and columns
df.shape

In [ ]:
df.columns

In [ ]:
#Methode um Modell performance (RMSE und cross validation) zu testen und den Entscheidungsbaum zu trainieren

In [ ]:
def model_performance(features, df, random_forest_model = RandomForestRegressor(random_state=42)):
    df = df.sample(frac=1, random_state=42)
    X, y = df[features], df['Price']
    scores = cross_val_score(random_forest_model, X, y, scoring="neg_root_mean_squared_error", cv=5)
    print('CV results RMSE:', np.round(scores))
    print('Mean RMSE:', np.mean(np.round(scores, 0)))

In [ ]:
# Data cleaning

In [ ]:
print('Total cars before data cleaning:', len(df))

# Remove missing values
df = df.dropna()

# Remove duplicates
df = df.drop_duplicates()

# Remove some 'extreme' values (Ausreisser entfernen)
df = df.loc[(df['Price'] >= 50000) & 
            (df['Price'] <= 10000000)]

# Kilometer-Ausreisser entfernen
df = df.loc[df['Kilometer'] <= 500000]

print('Total cars after data cleaning:', len(df))

In [ ]:
#Adding Features

In [ ]:
# Fahrzeugalter berechnen (2026 als Referenzjahr)
df['car_age'] = 2026 - df['Year']

In [ ]:
# Engine-Hubraum (cc) aus dem Text-Feld extrahieren
def parse_engine(val):
    if pd.isna(val): return None
    m = re.search(r'(\d+)', str(val))
    return int(m.group(1)) if m else None

df['engine_cc'] = df['Engine'].apply(parse_engine)

In [ ]:
# Max Power (bhp) aus dem Text-Feld extrahieren
def parse_power(val):
    if pd.isna(val): return None
    m = re.search(r'([\d.]+)', str(val))
    return float(m.group(1)) if m else None

df['max_power_bhp'] = df['Max Power'].apply(parse_power)

In [ ]:
# Automatikgetriebe als binäres Feature
df['is_automatic'] = (df['Transmission'] == 'Automatic').astype(int)
print('Total Automatic cars:', df['is_automatic'].sum())

In [ ]:
# Erstbesitzer als binäres Feature
df['is_first_owner'] = (df['Owner'] == 'First').astype(int)
print('Total First-Owner cars:', df['is_first_owner'].sum())

In [ ]:
#Eigene neue Features

In [ ]:
# Kilometer pro Jahr (Abnutzungsintensität)
df['km_per_year'] = round(df['Kilometer'] / df['car_age'].replace(0, 1), 2)

In [ ]:
# Leistung pro Hubraum (Motoreffizienz)
df['power_per_cc'] = round(df['max_power_bhp'] / df['engine_cc'], 4)

In [ ]:
# Luxusmarken-Flag
luxury_brands = ['Mercedes-Benz', 'BMW', 'Audi', 'Porsche']
df['is_luxury'] = df['Make'].isin(luxury_brands).astype(int)
print('Total Luxury cars:', df['is_luxury'].sum())

In [ ]:
# Elektro/Hybrid als binäres Feature (umweltfreundlich)
df['is_electric_hybrid'] = df['Fuel Type'].isin(['Electric', 'Hybrid']).astype(int)
print('Total Electric/Hybrid cars:', df['is_electric_hybrid'].sum())

In [ ]:
# Iteration 1

In [ ]:
df.columns

In [ ]:
features = ['car_age', 'Kilometer', 'engine_cc', 'max_power_bhp', 'is_automatic', 
            'is_first_owner', 'km_per_year', 'power_per_cc', 'is_luxury', 'is_electric_hybrid',
            'Seating Capacity', 'Fuel Tank Capacity']
model_performance(features, df)

In [ ]:
# train random_forest_model = RandomForestRegressor()
random_forest_model = RandomForestRegressor(random_state=42)

# Fit the model
random_forest_model.fit(df[features], df.Price)

cols = random_forest_model.feature_names_in_

# Derive feature importance from random forest
importances = random_forest_model.feature_importances_
std         = np.std([tree.feature_importances_ for tree in random_forest_model.estimators_], axis=0)
indices     = np.argsort(importances)[::-1]

# Barplot with feature importance
df_fi = pd.DataFrame({'features':cols,'importances': importances})
df_fi.sort_values('importances', inplace=True)
df_fi.plot(kind='barh', 
           y='importances',
           x='features', 
           color='darkred');

In [ ]:
# Linearregression Modell zu der Iteration 1

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# 1. Daten vorbereiten (X = Features, y = Zielvariable)
X = df[features]
y = df['Price']

# 3. Modell trainieren
model = LinearRegression().fit(X, y)

scores = cross_val_score(model, X, y, scoring="neg_root_mean_squared_error", cv=5)
print('CV results RMSE:', np.round(scores))
print('Mean RMSE:', np.mean(np.round(scores, 0)))

# 4. Ergebnis auswerten
r2 = model.score(X, y)
print(f"Modell-Genauigkeit (R^2): {r2:.2%}")

In [ ]:
#Fazit Iteration 1: Alle Features inkl. Fuel Tank Capacity und Seating Capacity

In [ ]:
# Iteration 2

In [ ]:
# Seating Capacity und Fuel Tank Capacity haben gemäss Feature-Importance-Plot geringe Relevanz
# Diese werden in Iteration 2 ausgeschlossen

In [ ]:
#Hier ist nun die Performance ohne die schwachen Features

In [ ]:
features_iteration2 = ['car_age', 'Kilometer', 'engine_cc', 'max_power_bhp', 'is_automatic', 
                        'is_first_owner', 'km_per_year', 'power_per_cc', 'is_luxury', 'is_electric_hybrid']
model_performance(features_iteration2, df, RandomForestRegressor(random_state=42))

In [ ]:
#Hier noch die Verteilung (Wichtigkeit der einzelnen Features)

In [ ]:
# train random_forest_model = RandomForestRegressor()
random_forest_model = RandomForestRegressor(random_state=42)

# Fit the model
random_forest_model.fit(df[features_iteration2], df.Price)

cols = random_forest_model.feature_names_in_

# Derive feature importance from random forest
importances = random_forest_model.feature_importances_
std         = np.std([tree.feature_importances_ for tree in random_forest_model.estimators_], axis=0)
indices     = np.argsort(importances)[::-1]

# Barplot with feature importance
df_fi = pd.DataFrame({'features':cols,'importances': importances})
df_fi.sort_values('importances', inplace=True)
df_fi.plot(kind='barh', 
           y='importances',
           x='features', 
           color='darkred');

In [ ]:
# Linearregression Modell zu der Iteration 2

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# 1. Daten vorbereiten (X = Features, y = Zielvariable)
X = df[features_iteration2]
y = df['Price']

# 3. Modell trainieren
model = LinearRegression().fit(X, y)

scores = cross_val_score(model, X, y, scoring="neg_root_mean_squared_error", cv=5)
print('CV results RMSE:', np.round(scores))
print('Mean RMSE:', np.mean(np.round(scores, 0)))

# 4. Ergebnis auswerten
r2 = model.score(X, y)
print(f"Modell-Genauigkeit (R^2): {r2:.2%}")

In [ ]:
# Fazit Iteration 2: Features reduziert auf die relevantesten 10

In [ ]:
# Iteration 3 (nur die 10 wichtigsten Features mit Hyperparameter-Tuning)

In [ ]:
# Die 10 wichtigsten Features gemäss dem Plot von oben

In [ ]:
top10_features = ['car_age', 'Kilometer', 'engine_cc', 'max_power_bhp', 'is_automatic',
                  'is_first_owner', 'km_per_year', 'power_per_cc', 'is_luxury', 'is_electric_hybrid']

In [ ]:
#Der Entscheidungsbaum: Hyperparameter definieren

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth':         [5, 10, 25],
    'n_estimators':      [100, 500, 1000]
}
grid_search = GridSearchCV(RandomForestRegressor(random_state=42), param_grid, cv=3,
scoring='neg_root_mean_squared_error', verbose = 2)
grid_search.fit(df[top10_features], df.Price)

# get best estimator:
grid_search.best_estimator_

In [ ]:
#Der Entscheidungsbaum nun nur mit den 10 wichtigsten Features

In [ ]:
# train random_forest_model = RandomForestRegressor() mit besten Hyperparametern
random_forest_model = RandomForestRegressor(max_depth=25, n_estimators=1000, random_state=42)

# Fit the model
random_forest_model.fit(df[top10_features], df.Price)

cols = random_forest_model.feature_names_in_

# Derive feature importance from random forest
importances = random_forest_model.feature_importances_
std         = np.std([tree.feature_importances_ for tree in random_forest_model.estimators_], axis=0)
indices     = np.argsort(importances)[::-1]

# Barplot with feature importance
df_fi = pd.DataFrame({'features':cols,'importances': importances})
df_fi.sort_values('importances', inplace=True)
df_fi.plot(kind='barh', 
           y='importances',
           x='features', 
           color='darkred');

In [ ]:
model_performance(top10_features, df, RandomForestRegressor(max_depth=25, n_estimators=1000, random_state=42))

In [ ]:
#Das LinearRegression Modell mit den 10 wichtigsten Features

In [ ]:
X = df[top10_features]
y = df['Price']

# 3. Modell trainieren
model = LinearRegression().fit(X, y)

scores = cross_val_score(model, X, y, scoring="neg_root_mean_squared_error", cv=5)
print('CV results RMSE:', np.round(scores))
print('Mean RMSE:', np.mean(np.round(scores, 0)))

# 4. Ergebnis auswerten
r2 = model.score(X, y)
print(f"Modell-Genauigkeit (R^2): {r2:.2%}")

In [ ]:
# Fazit Iteration 3: Performance konnte trotz weniger Features mit Hyperparameter gesteigert werden

In [ ]:
# Iteration 4

In [ ]:
# Iteration 4 – Verbesserungen gegenüber Iteration 3:
# 1. Engeres Ausreisser-Clipping (1. und 99. Perzentil)
# 2. Erweiterter Luxusmarken-Flag (mehr Marken)


In [ ]:
# NEU: Engeres Ausreisser-Clipping auf 1. und 99. Perzentil
p01 = df['Price'].quantile(0.01)
p99 = df['Price'].quantile(0.99)
df = df.loc[(df['Price'] >= p01) & (df['Price'] <= p99)]
print(f'Nach Percentile-Clipping (1%-99%): {len(df)} Zeilen')
print(f'Preis-Range: {p01:,.0f} – {p99:,.0f} INR')


In [ ]:
# NEU: Erweiterter Luxusmarken-Flag
# Iteration 3 enthielt nur 4 Marken; hier werden weitere Premium-/Luxusmarken ergänzt
luxury_brands_extended = [
    'Mercedes-Benz', 'BMW', 'Audi', 'Porsche',
    'Jaguar', 'Land Rover', 'Volvo', 'Lexus'
]
df['is_luxury'] = df['Make'].isin(luxury_brands_extended).astype(int)
print('Total Luxury cars (erweitert):', df['is_luxury'].sum())


In [ ]:
# Iteration 4: RandomForest mit denselben Hyperparametern wie Iteration 3
top10_features = ['car_age', 'Kilometer', 'engine_cc', 'max_power_bhp', 'is_automatic',
                  'is_first_owner', 'km_per_year', 'power_per_cc', 'is_luxury', 'is_electric_hybrid']

random_forest_model = RandomForestRegressor(max_depth=25, n_estimators=1000, random_state=42)
model_performance(top10_features, df, RandomForestRegressor(max_depth=25, n_estimators=1000, random_state=42))


In [ ]:
# Feature Importance – Iteration 4
random_forest_model.fit(df[top10_features], df['Price'])
df_fi = pd.DataFrame({'features': top10_features, 'importances': random_forest_model.feature_importances_})
df_fi.sort_values('importances', inplace=True)
df_fi.plot(kind='barh', y='importances', x='features', color='darkred');


In [ ]:
# Fazit Iteration 4: Engeres Clipping und mehr Luxusmarken verbessern den RMSE

In [ ]:
import pickle

In [ ]:
pickle.dump(random_forest_model, open('car_model.pkl','wb'))

In [ ]:
#Abschluss